# Zain Jordan Customer 360 AI Workshop  
## Class 5: Multi-Agent Customer Care Copilot

### Class Goal

In the previous classes, we built separate capabilities:

- **Class 1:** Python + SQLite database exploration
- **Class 2:** LangChain tools + single customer-care agent
- **Class 3:** Natural Language SQL Agent
- **Class 4:** RAG Assistant from telecom database content

In Class 5, we combine these building blocks into one practical system:

# Multi-Agent Customer Care Copilot

The goal is to show how multiple specialist agents can work together:

1. **Customer Data Agent** — gets customer-specific facts
2. **SQL Analyst Agent** — answers broader business questions from the database
3. **RAG Recommendation Agent** — finds plans, add-ons, campaigns, and customer-experience context
4. **Care Message Agent** — drafts professional customer-facing messages
5. **Supervisor Agent** — coordinates all agents and produces the final response

---

## Main Use Case

> Analyze customer 42.  
> Check customer profile, current plan, churn risk, complaints, support history, billing, usage, and relevant offers.  
> Recommend the next best action and draft a professional customer-care message.

---

## Important Boundary

This class focuses on:

- Multi-agent design
- Supervisor pattern
- Reusing tools, SQL agent, and RAG
- Customer-care workflow
- Capstone preparation

We are **not** implementing MCP yet. MCP will come in the next class.


# 1. Learning Outcomes

By the end of this notebook, participants should be able to:

1. Explain why multi-agent systems are useful.
2. Understand the supervisor pattern.
3. Build a Customer Data Agent using database tools.
4. Build a SQL Analyst Agent using the SQL agent pattern.
5. Build a RAG Recommendation Agent using database documents.
6. Build a Care Message Agent for professional response writing.
7. Combine all specialist agents under a Supervisor Agent.
8. Run a full customer-care analysis using the Zain Jordan database.
9. Connect the architecture to capstone project ideas.


# 2. Concept: One Agent vs Multi-Agent

| One Agent | Multi-Agent |
|---|---|
| One agent has many tools | Multiple specialist agents |
| Can become overloaded | Clear responsibility per agent |
| Harder to debug | Easier to test each part |
| Less modular | More reusable |
| Good for simple workflows | Better for complex workflows |

## Multi-Agent Idea

A real customer-care process usually involves multiple roles:

- Someone checks customer data
- Someone checks business trends
- Someone recommends offers
- Someone writes the customer message
- A supervisor combines everything

We mirror this with AI agents.


# 3. Architecture

```text
User Request
   ↓
Supervisor Agent
   ↓
Specialist Agents as Tools:
   ├── Customer Data Agent
   ├── SQL Analyst Agent
   ├── RAG Recommendation Agent
   └── Care Message Agent
   ↓
Final Customer Care Report
```

The supervisor does not need to know every detail.  
It coordinates the right specialist agents and combines the results.


# 4. Install Required Packages

Run this first in Google Colab.


In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langgraph pandas sqlalchemy


# 5. Import Libraries


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path
from collections import Counter

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


# 6. Set OpenAI API Key

In Google Colab:

1. Click the key icon on the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key.
4. Enable notebook access for the secret.


In [ ]:
try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


# 7. Upload or Locate the Zain Jordan Database

Upload the same database file used in previous classes:

`zain_customer_360_ai_demo.db`


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


# 8. Connect to SQLite Database


In [ ]:
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

db_path_obj = Path(DB_PATH).resolve()

print("Database path:", db_path_obj)
print("File exists:", db_path_obj.exists())

conn = sqlite3.connect(str(db_path_obj), check_same_thread=False)

tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


# 9. Initialize the Language Model

You can change `MODEL_NAME` depending on your available OpenAI model.


In [ ]:
from langchain.chat_models import init_chat_model

MODEL_NAME = "gpt-4.1-mini"

llm = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0
)

print("LLM initialized:", MODEL_NAME)


# 10. Helper Functions

These helpers will be reused by all agents.


In [ ]:
def df_to_text(df, max_rows=10):
    """Convert a pandas DataFrame into readable text for an AI tool."""
    if df is None or df.empty:
        return "No records found."
    return df.head(max_rows).to_string(index=False)


def extract_final_text(result):
    """Extract readable final text from a LangChain agent result."""
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\\n".join(parts)

    return str(last_message)


def run_agent(agent, question: str):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


# 11. Build Customer Database Functions

These are direct Python functions that read from the SQLite database.

Later, we convert them into LangChain tools.


In [ ]:
def fetch_customer_profile(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        c.gender,
        c.age_group,
        c.city,
        c.governorate,
        c.customer_type,
        c.customer_segment,
        c.preferred_language,
        c.status AS customer_status,
        a.account_type,
        a.account_status,
        a.credit_limit_jod
    FROM customers c
    LEFT JOIN accounts a
        ON c.customer_id = a.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_plan(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.msisdn,
        s.service_type,
        s.status AS subscription_status,
        p.plan_name,
        p.plan_category,
        p.monthly_fee_jod,
        p.data_allowance_gb,
        p.local_minutes,
        p.international_minutes,
        p.roaming_minutes,
        p.sms_allowance,
        p.technology,
        p.contract_months
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN plans p
        ON s.plan_id = p.plan_id
    WHERE c.customer_id = ?
    ORDER BY s.primary_subscription_flag DESC, s.activation_date DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_churn_risk(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        ch.score_month,
        ch.churn_score,
        ch.risk_level,
        ch.main_risk_reason,
        ch.recommended_action
    FROM customer_churn_scores ch
    JOIN customers c
        ON ch.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_complaints(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        resolved_date,
        compensation_amount_jod
    FROM complaints
    WHERE customer_id = ?
    ORDER BY complaint_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        resolution_status,
        resolution_time_minutes,
        customer_sentiment
    FROM support_interactions
    WHERE customer_id = ?
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        i.invoice_id,
        i.billing_period_start,
        i.billing_period_end,
        i.issue_date,
        i.due_date,
        i.total_amount_jod,
        i.payment_status,
        i.days_overdue
    FROM customers c
    JOIN accounts a
        ON c.customer_id = a.customer_id
    JOIN invoices i
        ON a.account_id = i.account_id
    WHERE c.customer_id = ?
    ORDER BY i.issue_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_usage_summary(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.service_type,
        COUNT(d.session_id) AS total_data_sessions,
        ROUND(SUM(d.data_used_mb) / 1024.0, 2) AS total_data_used_gb,
        ROUND(SUM(d.cost_jod), 2) AS total_data_cost_jod,
        MAX(d.session_start_time) AS last_data_session
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    LEFT JOIN data_usage_sessions d
        ON s.subscription_id = d.subscription_id
    WHERE c.customer_id = ?
    GROUP BY c.customer_id, c.full_name, s.subscription_id, s.service_type
    ORDER BY total_data_used_gb DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_value_segment(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        v.segment_month,
        v.arpu_jod,
        v.total_revenue_6m_jod,
        v.value_segment,
        v.lifetime_months
    FROM customer_value_segments v
    JOIN customers c
        ON v.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


# 12. Test Customer Database Functions

Always test normal functions before creating agents.


In [ ]:
print("PROFILE")
print(fetch_customer_profile(42))

print("\\nCHURN")
print(fetch_customer_churn_risk(42))

print("\\nVALUE")
print(fetch_customer_value_segment(42))


# 13. Convert Customer Functions into LangChain Tools


In [ ]:
from langchain.tools import tool

@tool
def get_customer_profile(customer_id: int) -> str:
    """Get customer profile, city, segment, language, account type, and account status for a Zain Jordan customer ID."""
    return fetch_customer_profile(customer_id)


@tool
def get_customer_plan(customer_id: int) -> str:
    """Get current subscriptions, mobile numbers, plan name, monthly fee, data allowance, minutes, and contract details for a customer ID."""
    return fetch_customer_plan(customer_id)


@tool
def get_customer_churn_risk(customer_id: int) -> str:
    """Get churn score, churn risk level, main risk reason, and recommended retention action for a customer ID."""
    return fetch_customer_churn_risk(customer_id)


@tool
def get_customer_complaints(customer_id: int, limit: int = 5) -> str:
    """Get recent customer complaints including complaint category, description, severity, status, and compensation amount."""
    return fetch_customer_complaints(customer_id, limit)


@tool
def get_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    """Get recent support interactions including channel, issue type, priority, resolution status, resolution time, and customer sentiment."""
    return fetch_customer_support_interactions(customer_id, limit)


@tool
def get_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    """Get recent invoice and billing summary including invoice dates, total amount, payment status, and days overdue for a customer ID."""
    return fetch_customer_billing_summary(customer_id, limit)


@tool
def get_customer_usage_summary(customer_id: int) -> str:
    """Get customer data usage summary including data sessions, total data used in GB, data cost, and last data session for a customer ID."""
    return fetch_customer_usage_summary(customer_id)


@tool
def get_customer_value_segment(customer_id: int) -> str:
    """Get customer value segment, ARPU, six-month revenue, and lifetime months for a Zain Jordan customer ID."""
    return fetch_customer_value_segment(customer_id)


# 14. Build Agent 1: Customer Data Agent

This specialist agent focuses only on customer-specific information.

It should not perform broad analytics.  
It should not recommend campaigns.  
It should only retrieve and summarize customer facts.


In [ ]:
from langchain.agents import create_agent

customer_data_tools = [
    get_customer_profile,
    get_customer_plan,
    get_customer_churn_risk,
    get_customer_complaints,
    get_customer_support_interactions,
    get_customer_billing_summary,
    get_customer_usage_summary,
    get_customer_value_segment,
]

customer_data_agent_prompt = """
You are the Customer Data Agent for Zain Jordan.

Your job:
- Retrieve customer-specific facts from the Zain Jordan Customer 360 database.
- Use tools whenever customer facts are needed.
- Do not guess.
- If data is missing, clearly say it is not available.

You can provide:
- customer profile
- current plan
- churn risk
- complaints
- support history
- billing summary
- usage summary
- value segment

Return a structured summary.
"""

customer_data_agent = create_agent(
    model=llm,
    tools=customer_data_tools,
    system_prompt=customer_data_agent_prompt,
)

print("Customer Data Agent created successfully.")


# 15. Test Customer Data Agent


In [ ]:
question = """
Get a complete customer 360 summary for customer 42.
Include profile, plan, churn risk, value segment, complaints, support history, billing, and usage.
"""

answer = run_agent(customer_data_agent, question)
print(answer)


# 16. Build Agent 2: SQL Analyst Agent

This specialist agent answers broader business questions from the database.

Examples:

- Which cities have the most high-risk churn customers?
- Which customer segments have the highest ARPU?
- What are the top complaint categories?
- Which campaigns converted best?


In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

db_uri = f"sqlite:///{db_path_obj}"
db = SQLDatabase.from_uri(db_uri)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

print("SQL tools:")
for t in sql_tools:
    print("-", t.name)


In [ ]:
sql_agent_prompt = """
You are the SQL Analyst Agent for Zain Jordan.

You answer broad telecom business questions using SQL.

Rules:
- Only use SELECT queries.
- Never use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, or CREATE.
- Do not modify the database.
- Inspect table schemas when needed.
- Limit results to 10 rows unless the user asks for more.
- Explain the business meaning of the result.
- Do not guess facts that are not in the database.

Common table guidance:
- Churn: customers, customer_churn_scores
- Revenue/value: customer_value_segments, invoices, payments, transactions
- Complaints/support: complaints, support_interactions
- Campaigns: campaigns, customer_campaign_responses
- Network: network_towers, network_events
"""

sql_analyst_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=sql_agent_prompt,
)

print("SQL Analyst Agent created successfully.")


# 17. Test SQL Analyst Agent


In [ ]:
question = "Which cities have the most high-risk churn customers? Show the top 5 and explain the business meaning."

answer = run_agent(sql_analyst_agent, question)
print(answer)


# 18. Build RAG Knowledge Base for Recommendation Agent

The RAG agent will use selected tables that contain recommendation or customer-experience context:

- plans
- addons
- campaigns
- complaints
- support_interactions
- customer_satisfaction


In [ ]:
from langchain_core.documents import Document

def build_plan_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        plan_id,
        plan_name,
        plan_category,
        service_type,
        monthly_fee_jod,
        data_allowance_gb,
        local_minutes,
        international_minutes,
        roaming_minutes,
        sms_allowance,
        technology,
        contract_months,
        data_carryover_flag,
        is_business_plan,
        status
    FROM plans
    WHERE status = 'Active';
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Plan ID: {row['plan_id']}
Plan Name: {row['plan_name']}
Category: {row['plan_category']}
Service Type: {row['service_type']}
Monthly Fee: {row['monthly_fee_jod']} JOD
Data Allowance: {row['data_allowance_gb']} GB
Local Minutes: {row['local_minutes']}
International Minutes: {row['international_minutes']}
Roaming Minutes: {row['roaming_minutes']}
SMS Allowance: {row['sms_allowance']}
Technology: {row['technology']}
Contract Months: {row['contract_months']}
Data Carryover: {bool(row['data_carryover_flag'])}
Business Plan: {bool(row['is_business_plan'])}

Use this plan information when recommending telecom plans to customers based on data usage, voice usage, roaming needs, technology preference, price sensitivity, and business or individual requirements.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "plans",
                "row_id": int(row["plan_id"]),
                "document_type": "plan",
                "plan_name": row["plan_name"],
                "plan_category": row["plan_category"],
            }
        ))
    return docs


def build_addon_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        addon_id,
        addon_name,
        addon_type,
        price_jod,
        validity_days,
        data_gb,
        minutes,
        sms,
        technology
    FROM addons;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Add-on ID: {row['addon_id']}
Add-on Name: {row['addon_name']}
Add-on Type: {row['addon_type']}
Price: {row['price_jod']} JOD
Validity Days: {row['validity_days']}
Data: {row['data_gb']} GB
Minutes: {row['minutes']}
SMS: {row['sms']}
Technology: {row['technology']}

Use this add-on information when recommending extra data, roaming, voice, SMS, or technology-specific bundles to customers.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "addons",
                "row_id": int(row["addon_id"]),
                "document_type": "addon",
                "addon_type": row["addon_type"],
            }
        ))
    return docs


def build_campaign_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        campaign_id,
        campaign_name,
        campaign_type,
        start_date,
        end_date,
        target_segment,
        offer_description,
        channel
    FROM campaigns;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Campaign ID: {row['campaign_id']}
Campaign Name: {row['campaign_name']}
Campaign Type: {row['campaign_type']}
Start Date: {row['start_date']}
End Date: {row['end_date']}
Target Segment: {row['target_segment']}
Offer Description: {row['offer_description']}
Channel: {row['channel']}

Use this campaign information when recommending promotional offers, campaign targeting, channel selection, and customer engagement ideas.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "campaigns",
                "row_id": int(row["campaign_id"]),
                "document_type": "campaign",
                "campaign_type": row["campaign_type"],
                "target_segment": row["target_segment"],
            }
        ))
    return docs


In [ ]:
def build_complaint_documents(conn, limit=150):
    df = pd.read_sql_query("""
    SELECT 
        complaint_id,
        customer_id,
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        compensation_amount_jod
    FROM complaints
    ORDER BY complaint_date DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Complaint ID: {row['complaint_id']}
Customer ID: {row['customer_id']}
Complaint Date: {row['complaint_date']}
Complaint Category: {row['complaint_category']}
Complaint Description: {row['complaint_description']}
Severity: {row['severity']}
Status: {row['status']}
Compensation Amount: {row['compensation_amount_jod']} JOD

Use this complaint information to identify customer pain points, service issues, complaint themes, and customer experience risks.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "complaints",
                "row_id": int(row["complaint_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "complaint",
                "complaint_category": row["complaint_category"],
                "severity": row["severity"],
            }
        ))
    return docs


def build_support_documents(conn, limit=150):
    df = pd.read_sql_query("""
    SELECT 
        interaction_id,
        customer_id,
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        resolution_status,
        resolution_time_minutes,
        customer_sentiment
    FROM support_interactions
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Support Interaction ID: {row['interaction_id']}
Customer ID: {row['customer_id']}
Interaction DateTime: {row['interaction_datetime']}
Channel: {row['channel']}
Reason Category: {row['reason_category']}
Issue Type: {row['issue_type']}
Priority: {row['priority']}
Resolution Status: {row['resolution_status']}
Resolution Time Minutes: {row['resolution_time_minutes']}
Customer Sentiment: {row['customer_sentiment']}

Use this support interaction information to understand support reasons, sentiment, support channels, issue types, and service quality.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "support_interactions",
                "row_id": int(row["interaction_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "support_interaction",
                "channel": row["channel"],
                "sentiment": row["customer_sentiment"],
            }
        ))
    return docs


def build_satisfaction_documents(conn, limit=150):
    df = pd.read_sql_query("""
    SELECT 
        survey_id,
        customer_id,
        survey_date,
        nps_score,
        csat_score,
        feedback_text,
        sentiment
    FROM customer_satisfaction
    ORDER BY survey_date DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Survey ID: {row['survey_id']}
Customer ID: {row['customer_id']}
Survey Date: {row['survey_date']}
NPS Score: {row['nps_score']}
CSAT Score: {row['csat_score']}
Feedback Text: {row['feedback_text']}
Sentiment: {row['sentiment']}

Use this satisfaction feedback to understand customer sentiment, common issues, loyalty signals, and improvement opportunities.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "customer_satisfaction",
                "row_id": int(row["survey_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "satisfaction_feedback",
                "sentiment": row["sentiment"],
            }
        ))
    return docs


In [ ]:
documents = (
    build_plan_documents(conn)
    + build_addon_documents(conn)
    + build_campaign_documents(conn)
    + build_complaint_documents(conn, limit=150)
    + build_support_documents(conn, limit=150)
    + build_satisfaction_documents(conn, limit=150)
)

print("Total documents:", len(documents))
Counter(doc.metadata["document_type"] for doc in documents)


# 19. Create Vector Store for RAG


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(documents)

print("Total chunks:", len(split_docs))

EMBEDDING_MODEL = "text-embedding-3-small"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore(embeddings)

document_ids = vector_store.add_documents(split_docs)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

print("Vector store created. Documents added:", len(document_ids))


# 20. Build RAG Answer Function


In [ ]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_table", "unknown")
        doc_type = doc.metadata.get("document_type", "unknown")
        row_id = doc.metadata.get("row_id", "unknown")

        formatted.append(
            f"[Document {i} | Source: {source} | Type: {doc_type} | Row ID: {row_id}]\\n{doc.page_content}"
        )
    return "\\n\\n".join(formatted)


def rag_answer_with_sources(question: str, k: int = 5) -> str:
    docs = retriever.invoke(question)[:k]
    context = format_docs(docs)

    prompt = f"""
You are the RAG Recommendation Agent for Zain Jordan.

Answer the question using only the retrieved context below.

Rules:
- Do not invent facts.
- If context is insufficient, clearly say what is missing.
- Focus on plans, add-ons, campaigns, support themes, complaints, and customer experience recommendations.
- Keep the answer structured and business-friendly.

Retrieved Context:
{context}

Question:
{question}

Answer:
"""
    response = llm.invoke(prompt)

    source_lines = []
    for doc in docs:
        source_lines.append(
            f"- {doc.metadata.get('source_table')} | {doc.metadata.get('document_type')} | Row ID: {doc.metadata.get('row_id')}"
        )

    return response.content + "\\n\\nRetrieved Sources:\\n" + "\\n".join(source_lines)


# 21. Test RAG Recommendation Function


In [ ]:
question = "Find relevant plans, add-ons, or campaigns for a high data usage customer with churn risk."

answer = rag_answer_with_sources(question)
print(answer)


# 22. Build Agent 3: RAG Recommendation Agent

This specialist agent focuses on recommendations and customer-experience context.


In [ ]:
@tool
def search_telecom_recommendations(question: str) -> str:
    """Search Zain Jordan telecom plans, add-ons, campaigns, complaints, support interactions, and satisfaction feedback using RAG."""
    return rag_answer_with_sources(question, k=5)


rag_recommendation_agent_prompt = """
You are the RAG Recommendation Agent for Zain Jordan.

Use the search_telecom_recommendations tool for:
- plan recommendations
- add-on recommendations
- campaign suggestions
- complaint themes
- support issue themes
- customer experience improvement recommendations

Do not guess.
Use retrieved context.
Return structured recommendations.
"""

rag_recommendation_agent = create_agent(
    model=llm,
    tools=[search_telecom_recommendations],
    system_prompt=rag_recommendation_agent_prompt,
)

print("RAG Recommendation Agent created successfully.")


# 23. Test RAG Recommendation Agent


In [ ]:
question = """
A customer has high churn risk and appears to need more data.
Find a relevant plan, add-on, or campaign recommendation.
"""

answer = run_agent(rag_recommendation_agent, question)
print(answer)


# 24. Build Agent 4: Care Message Agent

This agent does not need database access.

Its job is to convert analysis into a professional message.


In [ ]:
def care_message_agent_response(context: str) -> str:
    prompt = f"""
You are the Care Message Agent for Zain Jordan.

Your job is to write professional customer-care communication.

Use the context below to draft:
1. A short customer-facing message
2. A short internal note for the care team

Rules:
- Be polite, professional, and clear.
- Do not mention internal churn score directly to the customer.
- Do not overpromise compensation.
- Mention that the team will review or support the customer's concern.
- Keep the customer message under 120 words.

Context:
{context}

Output:
"""
    response = llm.invoke(prompt)
    return response.content


@tool
def draft_customer_care_message(context: str) -> str:
    """Draft a professional Zain Jordan customer-care message and internal follow-up note from customer analysis context."""
    return care_message_agent_response(context)


print("Care Message Agent function created.")


# 25. Test Care Message Agent


In [ ]:
sample_context = """
Customer 42 has high churn risk due to recent complaints and possible service dissatisfaction.
Recommended action: proactive retention call and review of plan/add-on options.
"""

print(draft_customer_care_message.invoke(sample_context))


# 26. Wrap Specialist Agents as Supervisor Tools

Now we expose each specialist agent as a tool.

The Supervisor Agent will call these tools.


In [ ]:
@tool
def customer_data_agent_tool(task: str) -> str:
    """Use the Customer Data Agent to retrieve customer-specific profile, plan, churn, complaints, support, billing, usage, and value details. Include the customer ID in the task."""
    return run_agent(customer_data_agent, task)


@tool
def sql_analyst_agent_tool(question: str) -> str:
    """Use the SQL Analyst Agent to answer broad telecom business questions from the Zain Jordan database using SQL."""
    return run_agent(sql_analyst_agent, question)


@tool
def rag_recommendation_agent_tool(question: str) -> str:
    """Use the RAG Recommendation Agent to find relevant plans, add-ons, campaigns, complaint themes, support themes, and customer experience recommendations."""
    return run_agent(rag_recommendation_agent, question)


@tool
def care_message_agent_tool(context: str) -> str:
    """Use the Care Message Agent to draft a professional customer-facing message and internal care-team note."""
    return care_message_agent_response(context)


supervisor_tools = [
    customer_data_agent_tool,
    sql_analyst_agent_tool,
    rag_recommendation_agent_tool,
    care_message_agent_tool,
]

print("Specialist agents wrapped as supervisor tools.")


# 27. Build Supervisor Agent

The Supervisor Agent coordinates the specialist agents.

It should:

1. Understand the request.
2. Call the Customer Data Agent for customer-specific facts.
3. Call the SQL Analyst Agent for broader trends if useful.
4. Call the RAG Recommendation Agent for plan/add-on/campaign/customer-experience recommendations.
5. Call the Care Message Agent to draft customer communication.
6. Combine everything into one final customer-care report.


In [ ]:
supervisor_prompt = """
You are the Supervisor Agent for a Zain Jordan Multi-Agent Customer Care Copilot.

Your job is to coordinate specialist agents and produce a complete customer-care report.

Available specialist tools:
1. customer_data_agent_tool
   - Use this for customer profile, plan, churn risk, value, complaints, support, billing, and usage.
2. sql_analyst_agent_tool
   - Use this for broader business trends, segment-level analysis, churn by city, complaint patterns, revenue insights.
3. rag_recommendation_agent_tool
   - Use this for plan, add-on, campaign, complaint-theme, support-theme, and customer-experience recommendations.
4. care_message_agent_tool
   - Use this to draft the final customer-facing message and internal note.

Rules:
- Use customer_data_agent_tool whenever the request includes a customer ID.
- Use rag_recommendation_agent_tool when a plan, add-on, campaign, offer, or customer-experience recommendation is needed.
- Use sql_analyst_agent_tool when broader business context or comparison is useful.
- Use care_message_agent_tool before giving the final customer-facing message.
- Do not invent customer facts.
- If data is missing, say it is not available.
- Keep the final answer structured, professional, and business-friendly.

Final answer format:
1. Customer Summary
2. Key Issue Diagnosis
3. Churn and Value Risk
4. Evidence from Data
5. Recommended Next Action
6. Suggested Offer / Plan / Campaign
7. Customer-Care Message
8. Internal Follow-Up Notes
"""

supervisor_agent = create_agent(
    model=llm,
    tools=supervisor_tools,
    system_prompt=supervisor_prompt,
)

print("Supervisor Agent created successfully.")


# 28. Main Demo: Full Multi-Agent Customer Care Copilot

This is the main Class 5 demo.


In [ ]:
main_question = """
Analyze customer 42.

I want:
1. Customer profile
2. Current plan
3. Churn risk
4. Value segment
5. Billing status
6. Complaints and support history
7. Usage behavior
8. Relevant plan, add-on, or campaign recommendation
9. A professional customer-care message
10. Internal follow-up notes for the care team
"""

answer = run_agent(supervisor_agent, main_question)
print(answer)


# 29. Demo 2: Retention-Focused Multi-Agent Request


In [ ]:
question = """
Customer 25 may be at risk of leaving.

Use the multi-agent copilot to:
- understand the customer's profile and risk
- check complaints and support history
- recommend a retention action
- suggest any relevant offer or campaign
- draft a customer message
"""

answer = run_agent(supervisor_agent, question)
print(answer)


# 30. Demo 3: Compare Two Customers


In [ ]:
question = """
Compare customer 25 and customer 42.

Which customer should the retention team prioritize first?
Use customer data, churn risk, value segment, complaints, and support history.
Then recommend the next action.
"""

answer = run_agent(supervisor_agent, question)
print(answer)


# 31. Demo 4: Add Business Context

This shows how the supervisor can ask the SQL Analyst Agent for broader trends.


In [ ]:
question = """
Analyze customer 42 and also check the broader churn trend by city.

Tell me whether the customer's city appears to have a wider churn-risk issue.
Then recommend what the care team should do.
"""

answer = run_agent(supervisor_agent, question)
print(answer)


# 32. Optional: Simple Visible Orchestration

For beginners, it is helpful to show a simple non-agent orchestration.

This function manually calls the components in sequence.

This makes the logic very clear.


In [ ]:
def simple_multi_agent_orchestration(customer_id: int):
    customer_context = run_agent(
        customer_data_agent,
        f"Get full customer 360 summary for customer {customer_id}."
    )

    recommendation_context = run_agent(
        rag_recommendation_agent,
        f"Based on this customer context, recommend a relevant plan, add-on, campaign, or care action:\\n{customer_context}"
    )

    care_message = care_message_agent_response(
        f"Customer Context:\\n{customer_context}\\n\\nRecommendation Context:\\n{recommendation_context}"
    )

    final_report = f"""
CUSTOMER DATA AGENT OUTPUT
{customer_context}

RAG RECOMMENDATION AGENT OUTPUT
{recommendation_context}

CARE MESSAGE AGENT OUTPUT
{care_message}
"""
    return final_report


print(simple_multi_agent_orchestration(42))


# 33. Exercise 1: Identify the Right Agent

For each request, decide which agent should be used.

| User Request | Best Agent |
|---|---|
| Who is customer 42? | Customer Data Agent |
| Which city has the most churn? | SQL Analyst Agent |
| Which offer suits heavy data usage? | RAG Recommendation Agent |
| Draft a message to the customer. | Care Message Agent |
| Analyze customer and recommend action. | Supervisor Agent |


# 34. Exercise 2: Analyze Different Customers

Try:

- customer 10
- customer 25
- customer 42
- customer 100

Compare:

- Who is most urgent?
- Who has billing risk?
- Who has complaint risk?
- Who needs retention?


In [ ]:
exercise_question = """
Analyze customer 10.

Check profile, plan, churn risk, value segment, billing, complaints, support history, usage, and relevant recommendations.
Then draft a customer-care message.
"""

answer = run_agent(supervisor_agent, exercise_question)
print(answer)


In [ ]:
exercise_question = """
Analyze customer 100.

Check profile, plan, churn risk, value segment, billing, complaints, support history, usage, and relevant recommendations.
Then draft a customer-care message.
"""

answer = run_agent(supervisor_agent, exercise_question)
print(answer)


# 35. Exercise 3: Improve the Agent Design

Discuss:

1. Should we add a separate Billing Agent?
2. Should we add a separate Network Agent?
3. Should the SQL Analyst and RAG Recommendation Agent remain separate?
4. Which agents are needed for your capstone project?


# 36. Exercise 4: Capstone Mapping

Choose one capstone idea and map the architecture.

Example:

## Churn Rescue Copilot

| Component | Purpose |
|---|---|
| Customer Data Agent | Get customer profile, churn, complaints, billing |
| SQL Analyst Agent | Find churn trends by city/segment |
| RAG Recommendation Agent | Suggest offers or campaigns |
| Care Message Agent | Draft retention message |
| Supervisor Agent | Combine final recommendation |

Now create the same mapping for your own project.


In [ ]:
capstone_question = """
Suggest five capstone ideas using this multi-agent architecture.

For each idea, mention:
1. Business problem
2. Target user
3. Required agents
4. Database tables likely used
5. Final demo output
"""

answer = run_agent(supervisor_agent, capstone_question)
print(answer)


# 37. Common Mistakes in Multi-Agent Systems

## Mistake 1: Too many agents too early

Start simple. Add agents only when responsibilities are truly different.

## Mistake 2: Unclear agent roles

Each agent should have a clear job.

## Mistake 3: Supervisor doing everything

The supervisor should coordinate, not do all specialist work.

## Mistake 4: Poor tool descriptions

The supervisor depends on tool descriptions to choose the right subagent.

## Mistake 5: No testing of subagents

Always test each specialist agent before combining them.

## Mistake 6: Mixing SQL and RAG without reason

Use SQL for calculations.  
Use RAG for recommendation and explanation.


# 38. What We Built Today

In Class 5, we built:

1. Customer Data Agent
2. SQL Analyst Agent
3. RAG Recommendation Agent
4. Care Message Agent
5. Supervisor Agent
6. Specialist agents wrapped as tools
7. Full Multi-Agent Customer Care Copilot
8. Simple visible orchestration function
9. Capstone mapping exercise

---

## Next Class

Class 6 will focus on:

# MCP for Enterprise AI Tool Integration

The next step is to expose selected tools through MCP so agents and applications can access them through a standard protocol.


# 39. Trainer Closing Script

Today, we connected everything from the previous classes.

We used:
- database tools from Class 2
- SQL analysis from Class 3
- RAG recommendations from Class 4
- message generation through a care agent
- a supervisor agent to coordinate everything

This is our first complete AI system.

The key idea is:

> A multi-agent system works best when each agent has a clear responsibility and the supervisor coordinates the final workflow.

In the next class, we will introduce MCP and expose selected tools through a standard enterprise integration pattern.
